In [ ]:
# %load_ext autoreload
# %autoreload 2

In [ ]:
import os
import subprocess
import time
from pathlib import Path


os.environ["PYOPENGL_PLATFORM"] = "osmesa"
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"

import matplotlib.pyplot as plt
import numpy as np
import pyrender
import trimesh
from tqdm.auto import tqdm


import jax
from jax.example_libraries import optimizers
import jax.numpy as jnp
from jax.scipy.spatial.transform import Rotation as Rot

import genmetaballs.fmb.fm_render as fm_render

# Import local utilities
from genmetaballs.fmb.utils import DegradeLR, get_camera_rays, image_grid

# Project paths - handle both running from project root or notebooks directory
PROJECT_ROOT = Path().resolve().parent
print(f"Project root: {PROJECT_ROOT}")
print(f"JAX version: {jax.__version__}")
print(f"JAX backend: {jax.default_backend().upper()}")
print(f"JAX devices: {jax.devices()}")


## Configuration

Hard-coded parameters from `examples/fmb_config.yaml`:


In [ ]:
# Model Parameters
NUM_MIXTURE = 40
gmm_init_scale = 1.0
rand_sphere_size = 30

# Rendering Parameters
num_views = 20
image_width = 64
image_height = 64
vfov_degrees = 45

# Optimization Parameters
Nepochs = 10
batch_size = int(800 *(image_height * image_width / (64 * 64)))
initial_lr = 0.1
opt_shape_scale = 2.2
clip_alpha = 3.0e-8

# Learning rate schedule
lr_decay_p_thresh = 0.5
lr_decay_window = 10
lr_decay_p_window = 5
lr_decay_slope_less = -1.0e-4
lr_decay_max_drops = 4

# Random Seed
random_seed = 42

# Input/Output
mesh_file = PROJECT_ROOT / "data/cow/cow.obj"

print("Configuration loaded:")
print(f"  Mixtures: {NUM_MIXTURE}")
print(f"  Views: {num_views}")
print(f"  Image size: {image_width}×{image_height}")
print(f"  Epochs: {Nepochs}")
print(f"  Batch size: {batch_size}")


## Load 3D Model and Setup Cameras


In [ ]:
# Load mesh
if not mesh_file.exists():
    subprocess.run(
        ["bash", str((PROJECT_ROOT / "scripts/data/download_cow.sh").resolve().absolute())]
    )

mesh_tri = trimesh.load(mesh_file)

# Mesh statistics
num_vertices = len(mesh_tri.vertices)
num_faces = len(mesh_tri.faces)
shape_scale = float(mesh_tri.vertices.std(0).mean()) * 3
center = np.array(mesh_tri.vertices.mean(0))
shape_scale_mul = opt_shape_scale / shape_scale

print(f"Model: {mesh_file.name}")
print(f"Vertices: {num_vertices:,}")
print(f"Faces: {num_faces:,}")
print(f"Scale factor: {shape_scale:.4f}")
print(f"Center: [{center[0]:.3f}, {center[1]:.3f}, {center[2]:.3f}]")

# Setup camera parameters
image_size = (image_height, image_width)
focal_length = 0.5 * image_size[0] / np.tan((np.pi / 180.0) * vfov_degrees / 2)
cx = (image_size[1] - 1) / 2
cy = (image_size[0] - 1) / 2

print("Camera setup:")
print(f"  Focal length: {focal_length:.2f} pixels")
print(f"  Principal point: ({cx:.1f}, {cy:.1f})")

# Generate random camera poses
np.random.seed(random_seed)
rand_quats = np.random.randn(num_views, 4)
rand_quats = rand_quats / np.linalg.norm(rand_quats, axis=1, keepdims=True)


## Render Reference Views


In [ ]:
# Render reference views using pyrender

mesh = pyrender.Mesh.from_trimesh(mesh_tri)
ref_colors = []
ref_depths = []
scene = pyrender.Scene()
scene.add(mesh)

trans = []
render_start = time.perf_counter()

for quat in tqdm(rand_quats, desc="Rendering reference views"):
    R = Rot.from_quat(quat).as_matrix()
    loc = np.array([0, 0, 3 * shape_scale]) @ R + center
    trans.append(loc)
    pose = np.vstack([np.vstack([R, loc]).T, np.array([0, 0, 0, 1])])

    light = pyrender.SpotLight(
        color=np.ones(3),
        intensity=50.0,
        innerConeAngle=np.pi / 16.0,
        outerConeAngle=np.pi / 6.0,
    )
    scene.add(light, pose=pose)

    camera = pyrender.IntrinsicsCamera(
        focal_length, focal_length, cx, cy, znear=0.1 * shape_scale, zfar=100 * shape_scale
    )
    scene.add(camera, pose=pose)

    r = pyrender.OffscreenRenderer(image_size[1], image_size[0])
    color, target_depth = r.render(scene)
    target_depth[target_depth == 0] = np.nan
    ref_colors.append(color)
    ref_depths.append(target_depth)

    for node in list(scene.light_nodes):
        scene.remove_node(node)
    for node in list(scene.camera_nodes):
        scene.remove_node(node)
    r.delete()

render_time = (time.perf_counter() - render_start) * 1000
print(f"✓ Rendered {num_views} views in {render_time:.1f} ms")

# Create target silhouettes
target_sil = (~np.isnan(ref_depths)).astype(np.float32)

# Display reference renders
image_grid(ref_colors, rows=4, cols=5, rgb=True)
plt.show()

image_grid(target_sil, rows=4, cols=5, rgb=False, cmap="Greys")
plt.suptitle("Reference Masks")
plt.show()


## Setup Fuzzy Metaballs Renderer


In [ ]:
# Get hyperparameters
hyperparams = fm_render.hyperparams
beta2 = jnp.float32(np.exp(hyperparams[0]))
beta3 = jnp.float32(np.exp(hyperparams[1]))

print(f"Mixture components: {NUM_MIXTURE}")
print(f"Hyperparameter β₂: {float(beta2):.4f}")
print(f"Hyperparameter β₃: {float(beta3):.4f}")
print(f"Total parameters: {NUM_MIXTURE * 13}")

# JIT compile render function
render_jit = jax.jit(fm_render.render_func_rays)


## Initialize Fuzzy Metaballs


In [ ]:
# Initialize random Gaussian cloud
np.random.seed(random_seed)
rand_mean = center + np.random.multivariate_normal(
    mean=[0, 0, 0], cov=1e-2 * np.identity(3) * shape_scale, size=NUM_MIXTURE
)
rand_weight_log = jnp.log(np.ones(NUM_MIXTURE) / NUM_MIXTURE) + jnp.log(gmm_init_scale)
rand_prec = jnp.array([np.identity(3) * rand_sphere_size / shape_scale for _ in range(NUM_MIXTURE)])

print("Initialization: Random Gaussians near center")
print(f"Mean position: {center} ± {np.sqrt(1e-2 * shape_scale):.4f}")

# Setup camera rays
height, width = image_size
pixel_list = (
    (np.array(np.meshgrid(np.arange(width), height - np.arange(height) - 1, [0]))[:, :, :, 0])
    .reshape((3, -1))
    .T
)
camera_rays = get_camera_rays(focal_length, focal_length, cx, cy, pixel_list)
cameras_list = []
for tran, quat in zip(trans, rand_quats, strict=False):
    R = Rot.from_quat(quat).as_matrix()
    camera_rays2 = camera_rays @ R
    t = np.tile(tran[None], (camera_rays2.shape[0], 1))
    rays_trans = np.stack([camera_rays2, t], 1)
    cameras_list.append(rays_trans)

print(f"Camera rays: {len(cameras_list)} views × {camera_rays.shape[0]:,} rays/view")


## Benchmark Initial Forward Pass


In [ ]:
# Warmup JIT compilation
print("Warming up forward pass JIT compilation...")
for _ in range(3):
    _ = render_jit(
        rand_mean, rand_prec, rand_weight_log, cameras_list[0], beta2 / shape_scale, beta3
    )
print("✓ Forward pass JIT warmup complete")

# Benchmark forward passes
print(f"\nRunning {len(cameras_list)} forward passes...")
alpha_results_rand = []
alpha_results_rand_depth = []
forward_times = []

for camera_rays in cameras_list:
    t_start = time.perf_counter()
    est_depth, est_alpha, est_norm, est_w = render_jit(
        rand_mean, rand_prec, rand_weight_log, camera_rays, beta2 / shape_scale, beta3
    )
    est_alpha.block_until_ready()
    t_elapsed = (time.perf_counter() - t_start) * 1000
    forward_times.append(t_elapsed)

    alpha_results_rand.append(est_alpha.reshape(image_size))
    est_depth = np.array(est_depth)
    est_depth[est_alpha < 0.5] = np.nan
    alpha_results_rand_depth.append(est_depth.reshape(image_size))

avg_forward = np.mean(forward_times)
print("\n📊 Forward Pass Statistics:")
print(f"   Mean: {avg_forward:.3f} ms/frame ± {np.std(forward_times):.3f} ms")
print(f"   Total: {sum(forward_times):.1f} ms for {num_views} frames")

# Display initial renderings
image_grid(alpha_results_rand, rows=4, cols=5, rgb=False, cmap="Greys")
plt.suptitle("Random Init Masks")
plt.show()


## Setup Optimization


In [ ]:
# Define objective function
def objective(params, true_alpha):
    means, prec, weights_log, camera_rays, beta2, beta3 = params
    render_res = render_jit(means, prec, weights_log, camera_rays, beta2, beta3)
    est_alpha = render_res[1]
    est_alpha = jnp.clip(est_alpha, clip_alpha, 1 - clip_alpha)
    mask_loss = -((true_alpha * jnp.log(est_alpha)) + (1 - true_alpha) * jnp.log(1 - est_alpha))
    return mask_loss.mean()


grad_render3 = jax.jit(jax.value_and_grad(objective))

# Prepare data
all_cameras = jnp.array(cameras_list).reshape((-1, 2, 3))
all_sils = jnp.array(target_sil.ravel()).astype(jnp.float32)
Niter_epoch = int(np.ceil(len(all_cameras) / batch_size))

print("Optimizer: Adam with adaptive learning rate")
print(f"Initial LR: {initial_lr}")
print(f"Epochs: {Nepochs}")
print(f"Batch size: {batch_size} rays")
print(f"Iterations/epoch: {Niter_epoch}")

# Setup optimizer
vecM = jnp.array([[1, 1, 1], [shape_scale_mul, shape_scale_mul, shape_scale_mul]])[None]


def irc(x):
    return int(round(x))


adjust_lr = DegradeLR(
    initial_lr,
    lr_decay_p_thresh,
    irc(Niter_epoch * 0.4),
    irc(lr_decay_p_window),
    lr_decay_slope_less,
    lr_decay_max_drops,
)
opt_init, opt_update, opt_params = optimizers.adam(adjust_lr.step_func)
tmp = [rand_mean * shape_scale_mul, rand_prec / shape_scale_mul, rand_weight_log]
opt_state = opt_init(tmp)

# Warmup gradient computation
print("\nWarming up backward pass JIT compilation...")
p = opt_params(opt_state)
idx_sample = jnp.array(list(range(min(batch_size, len(all_cameras)))))

for _ in range(3):
    val, g = grad_render3(
        [p[0], p[1], p[2], vecM * all_cameras[idx_sample], beta2 / opt_shape_scale, beta3],
        all_sils[idx_sample],
    )
    jax.tree_util.tree_map(lambda x: x.block_until_ready(), g)

print("✓ Backward pass JIT warmup complete")


## Run Optimization


In [ ]:
rand_idx = np.arange(len(all_cameras))
losses = []
done = False
iteration_count = 0
backward_times = []

opt_start_time = time.perf_counter()

for i in range(Nepochs):
    np.random.shuffle(rand_idx)
    rand_idx_jnp = jnp.array(rand_idx)

    epoch_start = time.perf_counter()

    for j in range(Niter_epoch):
        p = opt_params(opt_state)
        idx = jax.lax.dynamic_slice(rand_idx_jnp, [j * batch_size], [batch_size])

        # Forward + backward pass
        t_start = time.perf_counter()
        val, g = grad_render3(
            [p[0], p[1], p[2], vecM * all_cameras[idx], beta2 / opt_shape_scale, beta3],
            all_sils[idx],
        )
        jax.tree_util.tree_map(lambda x: x.block_until_ready(), g)
        t_elapsed = (time.perf_counter() - t_start) * 1000
        backward_times.append(t_elapsed)

        opt_state = opt_update(i, g[:3], opt_state)
        val = float(val)
        losses.append(val)
        iteration_count += 1

        if adjust_lr.add(val):
            done = True
            break

    epoch_time = (time.perf_counter() - epoch_start) * 1000
    print(f"Epoch {i + 1:2d}/{Nepochs} | Loss: {losses[-1]:.4f} | Time: {epoch_time:6.1f} ms")

    if done:
        print("\n✓ Early stopping triggered")
        break

opt_total_time = (time.perf_counter() - opt_start_time) * 1000
avg_backward = np.mean(backward_times)

print("\n📊 Optimization Complete:")
print(f"   Total time: {opt_total_time:.1f} ms ({opt_total_time / 1000:.2f} seconds)")
print(f"   Total iterations: {iteration_count}")
print(f"   Final loss: {losses[-1]:.6f}")
print(f"   Initial loss: {losses[0]:.6f}")
print(f"   Loss reduction: {(1 - losses[-1] / losses[0]) * 100:.1f}%")
print(f"   Avg backward time: {avg_backward:.3f} ms/batch")

# Plot convergence
plt.figure(figsize=(10, 6))
plt.title("Convergence Plot - Shape from Silhouette", fontsize=14, fontweight="bold")
plt.plot(losses, marker=".", lw=0, ms=5, alpha=0.5, color="#2196F3")
plt.xlabel("Iteration", fontsize=12)
plt.ylabel("Binary Cross-Entropy Loss", fontsize=12)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## Render Final Results


In [ ]:
# Get final parameters
final_means, final_precs, final_weight_logs = opt_params(opt_state)
final_means /= shape_scale_mul
final_precs *= shape_scale_mul

# _i, _n = 4, 1
# final_means = final_means[_i:_i+_n]
# final_precs = final_precs[_i:_i+_n]
# final_weight_logs = final_weight_logs[_i:_i+_n]

# Render final results
alpha_results_final = []
alpha_results_depth = []

for camera_rays in cameras_list:
    est_depth, est_alpha, est_norms, est_w = render_jit(
        final_means, final_precs, final_weight_logs, camera_rays, beta2 / shape_scale, beta3
    )
    alpha_results_final.append(est_alpha.reshape(image_size))

    est_depth = np.array(est_depth)
    est_depth[est_alpha < 0.5] = np.nan
    alpha_results_depth.append(est_depth.reshape(image_size))

# Display results
image_grid(target_sil, rows=4, cols=5, rgb=False)
plt.suptitle("Reference Masks", fontsize=14, fontweight="bold")
plt.show()

image_grid(alpha_results_final, rows=4, cols=5, rgb=False)
plt.suptitle("Final Optimized Masks", fontsize=14, fontweight="bold")
plt.show()

# Depth comparison
vmin = np.nanmin(np.array(ref_depths))
vmax = np.nanmax(np.array(ref_depths))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
im1 = ax1.imshow(alpha_results_depth[3], vmin=vmin, vmax=vmax, cmap="viridis")
ax1.set_title("Estimated Depth (View 3)", fontsize=12, fontweight="bold")
ax1.axis("off")
plt.colorbar(im1, ax=ax1)

im2 = ax2.imshow(ref_depths[3], vmin=vmin, vmax=vmax, cmap="viridis")
ax2.set_title("Ground Truth Depth (View 3)", fontsize=12, fontweight="bold")
ax2.axis("off")
plt.colorbar(im2, ax=ax2)

plt.tight_layout()
plt.show()

image_grid(alpha_results_depth, rows=4, cols=5, rgb=False, vmin=vmin, vmax=vmax)
plt.suptitle("Final Depth Maps - All Views", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

End of original FMB demo notebook

----

## GenMetaBalls demo


In [ ]:
@jax.jit
def cov_to_isostds_and_quaternion(cov):
    """
    Convert a 3D Gaussian's covariance matrix to an isotropic stds vector and a rotation quaternion.

    Args:
    - cov: (3, 3) covariance matrix

    Returns:
    - vars: (3,) array of isotropic variances
    - quat: (4,) quaternion
    """
    eigvals, eigvecs = jnp.linalg.eigh(cov)

    # Ensure positive eigenvalues (numerical stability)
    vars_ = jnp.maximum(eigvals, 0)

    # Ensure deterministic eigenvector orientation
    for i in range(3):
        eigvecs = eigvecs.at[:, i].set(jnp.where(eigvecs[0, i] < 0, -eigvecs[:, i], eigvecs[:, i]))

    # Ensure proper rotation matrix (determinant +1)
    eigvecs = eigvecs.at[:, 0].set(
        jnp.where(jnp.linalg.det(eigvecs) < 0, -eigvecs[:, 0], eigvecs[:, 0])
    )

    # Convert rotation matrix to quaternion
    quat = Rot.from_matrix(eigvecs).as_quat()

    return vars_, quat

In [ ]:
from genmetaballs.core import (
    FMB,
    Intrinsics,
    ThreeParameterBlender,
    ZeroParameterConfidence,
    geometry,
    make_fmb_scene_from_values,
    render_fmbs,
    make_image,
    dim3
)

Pose, Vec3D, Rotation = geometry.Pose, geometry.Vec3D, geometry.Rotation

fmbs = []
for (final_mean, final_prec) in zip(final_means, final_precs):
    prec = final_prec @ final_prec.T
    stds, quat = cov_to_isostds_and_quaternion(jnp.linalg.inv(prec))
    pose = Pose.from_components(Rotation.from_quat(*quat), Vec3D(*final_mean))
    fmbs.append(FMB(pose, *stds))

scene = make_fmb_scene_from_values(fmbs, list(final_weight_logs), device="gpu")
blender_obj = ThreeParameterBlender(beta1=beta3, beta2=beta2, eta=shape_scale)
confidence_obj = ZeroParameterConfidence()
image = make_image(height, width, "gpu")

# Pre-compute intrinsics (same for all views)
intr = Intrinsics(fx=focal_length, fy=focal_length, cx=cx, cy=cy, width=width, height=height)

# Pre-compute all camera extrinsics
all_extrs = []
for camera_num in range(len(cameras_list)):
    extr = Pose.from_components(
        rot=Rotation.from_quat(*rand_quats[camera_num]).inv(),
        tran=Vec3D(*trans[camera_num]),
    )
    all_extrs.append(extr)

gmb_depths = []
gmb_confidences = []
render_times = []

iter_mult = 1000

for camera_num in tqdm(range(len(cameras_list))):
    for iter_idx in range(iter_mult):

        # if camera_num != 0:
        #     continue

        extr = all_extrs[camera_num]
        
        start_time = time.time_ns()
        render_fmbs(
            scene,
            blender=blender_obj,
            confidence=confidence_obj,
            intr=intr,
            extr=extr,
            img=image,
            grid_size=dim3(4, 4),
            block_size=dim3(16, 16),
            kernel_id=0,
            block=True
        )
        # _depth_img = image.as_view().depth.as_jax().block_until_ready()
        end_time = time.time_ns()
        render_times.append((end_time - start_time) / 1e3)  # microseconds
        
        if iter_idx == 0:
            img_view = image.as_view()

            depth_image = jnp.copy(img_view.depth.as_jax())
            confidence = jnp.copy(img_view.confidence.as_jax())
            gmb_depths.append(depth_image)
            gmb_confidences.append(confidence)

total_time_us = sum(render_times)
import numpy as np
stddev_time_us = np.std(render_times)
print(f"Time taken for {len(cameras_list) * iter_mult} views: {total_time_us:.2f} microseconds")
print("FPS: ", (len(cameras_list) * iter_mult) / total_time_us * 1e6)
print(f"Average render time per view: {total_time_us / len(cameras_list) / iter_mult:.2f} microseconds")
print(f"Standard deviation of render time per view: {stddev_time_us:.2f} microseconds")


In [ ]:
# Benchmark render_jit in the same manner as render_fmbs (FIXED: using trained parameters)
print("\n" + "="*50)
print("BENCHMARKING render_jit")
print("="*50)

jax_render_times = []
jax_depths = []
jax_alphas = []

for camera_num in range(len(cameras_list)):
    for iter_idx in range(iter_mult):
        
        camera_rays = cameras_list[camera_num]

        beta2_div_shape_scale = beta2 / shape_scale
        
        start_time = time.time_ns()
        est_depth, est_alpha, est_norm, est_w = render_jit(
            final_means, final_precs, final_weight_logs, camera_rays, beta2_div_shape_scale, beta3
        )
        est_alpha.block_until_ready()
        end_time = time.time_ns()
        jax_render_times.append((end_time - start_time) / 1e3)  # microseconds
        
        if iter_idx == 0:
            jax_depths.append(est_depth.reshape(image_size))
            jax_alphas.append(est_alpha.reshape(image_size))

total_jax_time_us = sum(jax_render_times)
stddev_jax_time_us = np.std(jax_render_times)
print(f"Time taken for {len(cameras_list) * iter_mult} views: {total_jax_time_us:.2f} microseconds")
print("FPS: ", (len(cameras_list) * iter_mult) / total_jax_time_us * 1e6)
print(f"Average render time per view: {total_jax_time_us / len(cameras_list) / iter_mult:.2f} microseconds")
print(f"Standard deviation of render time per view: {stddev_jax_time_us:.2f} microseconds")

# Compare performance
print("\n" + "="*50)
print("PERFORMANCE COMPARISON")
print("="*50)
gmb_avg_time = total_time_us / len(cameras_list) / iter_mult
jax_avg_time = total_jax_time_us / len(cameras_list) / iter_mult
speedup = jax_avg_time / gmb_avg_time

print(f"GenMetaBalls average: {gmb_avg_time:.2f} μs")
print(f"JAX render_jit average: {jax_avg_time:.2f} μs")
print(f"Speedup (JAX/GMB): {speedup:.2f}x")
if speedup > 1:
    print(f"GenMetaBalls is {speedup:.2f}x faster than JAX")
else:
    print(f"JAX is {1/speedup:.2f}x faster than GenMetaBalls")


In [ ]:
fig, axes = plt.subplots(20, 4, figsize=(16, 100))
# fig.suptitle("All Views (0-19)", fontsize=16, fontweight="bold")

for VIEW_IDX in range(20):
    ax0, ax1, ax2, ax3 = axes[VIEW_IDX]
    
    # FIXED: Use FILTERED CUDA depth (not raw) for fair comparison
    depth_image = jnp.where(gmb_confidences[VIEW_IDX] < 0.5, jnp.nan, gmb_depths[VIEW_IDX])
    im0 = ax0.imshow(depth_image, vmin=vmin, vmax=vmax, cmap="viridis")
    ax0.set_title("GenMetaBalls (View {})".format(VIEW_IDX), fontsize=10, fontweight="bold")
    ax0.axis("off")
    plt.colorbar(im0, ax=ax0)

    im1 = ax1.imshow(alpha_results_depth[VIEW_IDX], vmin=vmin, vmax=vmax, cmap="viridis")
    ax1.set_title("FMB-JAX (View {})".format(VIEW_IDX), fontsize=10, fontweight="bold")
    ax1.axis("off")
    plt.colorbar(im1, ax=ax1)

    im2 = ax2.imshow(ref_depths[VIEW_IDX], vmin=vmin, vmax=vmax, cmap="viridis")
    ax2.set_title("GT Depth (View {})".format(VIEW_IDX), fontsize=10, fontweight="bold")
    ax2.axis("off")
    plt.colorbar(im2, ax=ax2)

    # FIXED: Compare FILTERED CUDA vs FILTERED JAX (both at same processing stage)
    # Both depths are filtered (NaN where confidence/alpha < 0.5)
    gmb_depth_filtered = jnp.where(gmb_confidences[VIEW_IDX] < 0.5, jnp.nan, gmb_depths[VIEW_IDX])
    diff = gmb_depth_filtered - alpha_results_depth[VIEW_IDX]
    im3 = ax3.imshow(diff, cmap="RdBu_r")
    ax3.set_title("Diff (GMB - FMB-JAX) (View {})".format(VIEW_IDX), fontsize=10, fontweight="bold")
    ax3.axis("off")
    plt.colorbar(im3, ax=ax3)

plt.tight_layout()
plt.show()
plt.close()


In [ ]:
fig, axes = plt.subplots(20, 3, figsize=(18, 100))
# fig.suptitle("All Views (0-19)", fontsize=16, fontweight="bold")

for VIEW_IDX in range(20):
    ax0, ax1, ax2 = axes[VIEW_IDX]
    
    im0 = ax0.imshow(gmb_confidences[VIEW_IDX], vmin=0, vmax=1)
    ax0.set_title("GenMetaBalls Confidence (View {})".format(VIEW_IDX), fontsize=10, fontweight="bold")
    ax0.axis("off")
    plt.colorbar(im0, ax=ax0)

    im1 = ax1.imshow(alpha_results_final[VIEW_IDX], vmin=0, vmax=1, cmap="viridis")
    ax1.set_title("FMB-JAX Alpha (View {})".format(VIEW_IDX), fontsize=10, fontweight="bold")
    ax1.axis("off")
    plt.colorbar(im1, ax=ax1)

    diff = gmb_confidences[VIEW_IDX] - alpha_results_final[VIEW_IDX]
    im2 = ax2.imshow(diff, cmap="RdBu_r")
    ax2.set_title("Diff (View {})".format(VIEW_IDX), fontsize=10, fontweight="bold")
    ax2.axis("off")
    plt.colorbar(im2, ax=ax2)

plt.tight_layout()
plt.show()
plt.close()

In [ ]:
fmbs = [scene[i][0] for i in range(len(scene))]
log_weights = [scene[i][1] for i in range(len(scene))]

In [ ]:
unnorm_probs = np.exp(np.array(log_weights))
norm_probs = unnorm_probs / np.sum(unnorm_probs)
norm_probs_max_scaled = norm_probs/norm_probs.max()
norm_probs_max_scaled

In [ ]:
vec3d_to_np = lambda v: np.array([v.x, v.y, v.z])

import rerun as rr
from rerun.components import ViewCoordinates

rr.init("GenMetaBalls", spawn=False)
rr.connect_grpc()
rr.log("/", rr.Clear(recursive=True))

EXTEND_RAY_DIRS = 0.0

centers = np.array([[fmb.pose.tran.x, fmb.pose.tran.y, fmb.pose.tran.z] for fmb in fmbs])
extents = np.array([np.sqrt(fmb.extent) for fmb in fmbs])
quats = np.array([fmb.pose.rot.quat for fmb in fmbs])

# cam_ray_dirs = []
# for y in range(camera.height):
#     for x in range(camera.width):
#         cam_ray_dirs.append(vec3d_to_np(camera.get_ray_direction(x, y)))
# cam_ray_dirs = np.array(cam_ray_dirs)
# cam_ray_dirs = camera_rays

# Generate random colors for each ellipsoid
np.random.seed(42)
colors = np.random.randint(0, 256, size=(len(fmbs), 3))

rr.log("world/metaballs",
    rr.Ellipsoids3D(
        centers = centers,
        half_sizes = extents,
        quaternions = quats,
        colors=colors,
        fill_mode="Solid",
    ))

# rr.log("world/camera", rr.Pinhole(focal_length = camera.fx,
#                             principal_point = (camera.cx, camera.cy),
#                             width = camera.width,
#                             height = camera.height,
#                             image_plane_distance = 1.0,
#                             camera_xyz = ViewCoordinates.RUB
#                             ))
# rr.log("world/camera", rr.Image(alpha_image_rgb))
# rr.log("world/rays", rr.Arrows3D(
#     radii=0.01,
#     colors=[128,128,128],
#     vectors=cam_ray_dirs * EXTEND_RAY_DIRS))

# for ray_count in range(len(cam_ray_dirs)):
#     rr.set_time(timeline="ray_count", sequence=ray_count)
#     rr.log("world/rays", rr.Arrows3D(
#         radii=0.01,
#         colors=[128,128,128],
#         vectors=cam_ray_dirs[:ray_count] * EXTEND_RAY_DIRS))


